# Impact Labelling

This notebook serves as guidelines to label qualitative and quantitative impact information \
Other guidelines can be found here : https://docs.google.com/document/d/1SZk7cou4R6yV44UAxehUqOdqndOFiyHbUCUptlwkk_w/edit?tab=t.0#heading=h.2s6r53ihrgdp

Recommandations
- Work at the sentence level. Each time you identify one or several impacts in a sentence, extract all the information from it. Then continue looping over text sentence
- Information on the location can be present in different sentences

In [22]:
import pandas as pd
import json
from collections import Counter
import numpy as np
import pandas as pd
import seaborn as sns
import spacy
import re
import pycountry
from src.text_processing_functions import *
from src.LLM_functions import *
import copy as cp

from src.data import *

In [ ]:
### Paths
DATA_IN_JSONS = "..." #### CHANGE
DATA_LABELLED = "..." #### CHANGE 

In [2]:
#Load data
file_path = DATA_IN_JSONS +'all_ifrc_reports_info_processed_extended_format_nb_std_units.json'

# Open and read the JSON file
with open(file_path, 'r') as json_file:
    filtered_reports = json.load(json_file)
filtered_reports = pd.DataFrame(filtered_reports)

## Report labelling

In [3]:
# select reports to be labelled
appealCode_list = []  #### UPDATE HERE WITH THE LIST OF APPEAL CODE THAT NEED TO BE LABELLED

reports_to_label = filtered_reports.where(filtered_reports.appealCode.isin(appealCode_list)).dropna()

#convert dates
reports_to_label.date = pd.to_datetime(reports_to_label.date, dayfirst=True)

#select most recent reports
reports_to_label = reports_to_label.groupby('appealCode').apply(lambda x: x.sort_values('date', ascending=False).head(1))

#check that everything is there
print(f"Number appealCodes: {len(appealCode_list)}, number reports: {len(reports_to_label)}")

Number appealCodes: 13, number reports: 13


In [8]:
for groupname, group in reports_to_label.groupby("origType"):
    if len(group) > 1:
        print(f"{groupname}: {len(group)}")

In [14]:
labelled_impact_reports_dict = {} # dict to store labelled reports
labelled_impact_reports_dict_new_haz = {} # dict to store labelled reports
#### EMPTY DICT STRUCTURE, 
#### YOU CAN COPY THE FOLLOWING STRUCTURE WHEN CREATING A NEW IMPACT LINE
# labelled_impact_reports_dict["appealCode"]=[
#     {"reportDate": None,
#      "impactSubtypes" : None,
#      "impactValue" : None, 
#      "impactUnit" : None, 
#      "impactAnnotation" : None,
#      "country" : None,
#      "location" : None,
#      "locationAnnotation" : None,
#      "startYear" : None,
#      "startMonth" : None,
#      "startDay" : None,
#      "endYear" : None,
#      "endMonth" : None,
#      "endDay" : None,
#      "hazardType" : None,
#     },
# ]

In [13]:
i=0
print(f"{reports_to_label.iloc[i].appealCode}: {reports_to_label.iloc[i].date}")
reports_to_label.iloc[i].nathaz_text

MDRBD015: 2016-03-04 00:00:00


['The Government district level Dform data immediately after the disaster indicated many houses were flattened or under water, trees uprooted, and power supplies and communication systems disrupted in some places.',
 'Crops were damaged and shrimp projects flooded.',
 'Due to the impact of the cyclonic storm, heavy to very heavy rainfall triggered in southern Bangladesh widespread flooding.',
 'Consequently the lives and livelihoods of the people of those areas were further worsened.',
 'Emergency appeal operations update Bangladesh Cyclone Komen 2 P a g e A Need Assessment Working Group NAWG was formed to identify the damage and needs of all these areas affected by Cyclone Komen and subsequent flooding.',
 'This assessment was commissioned by the Humanitarian Coordination Task Team HCTT and covered ten districts.',
 'The cumulative effect of the floods coming after Cyclone Komen increased the affected population to 2.6 million people.',
 'The impact of these events was felt most acute

In [17]:
labelled_impact_reports_dict_new_haz["MDRBD015"]=[
    {"reportDate": "2015-09-16",
     "impactSubtypes" : "Residential Buildings",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactAnnotation" : ['The Government district level ‘D-form’ data immediately after the disaster indicated many houses were flattened or under water, trees uprooted, and power supplies and communication systems disrupted in some places.'],
     "country" : "Bangladesh",
     "location" : None,
     "locationAnnotation" : None,
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardType" : ["Flood", "Tropical storms"],
    },
    {"reportDate": "2015-09-16",
     "impactSubtypes" : "IT and Communication Infrastructure",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactAnnotation" : ['The Government district level ‘D-form’ data immediately after the disaster indicated many houses were flattened or under water, trees uprooted, and power supplies and communication systems disrupted in some places.'],
     "country" : "Bangladesh",
     "location" : None,
     "locationAnnotation" : None,
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardType" : ["Flood", "Tropical storms"],
    },
    {"reportDate": "2015-09-16",
     "impactSubtypes" : "Agriculture",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactAnnotation" : ['Crops were damaged and shrimp projects flooded.'],
     "country" : "Bangladesh",
     "location" : None,
     "locationAnnotation" : None,
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardType" : ["Flood", "Tropical storms"],
    },
    {"reportDate": "2015-09-16",
     "impactSubtypes" : "Affected People",
     "impactValue" : 2600000, 
     "impactUnit" : "people", 
     "impactAnnotation" : ['The cumulative effect of the floods coming after Cyclone Komen increased the affected population to 2.6 million people.'],
     "country" : "Bangladesh",
     "location" : None,
     "locationAnnotation" : None,
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardType" : ["Flood", "Tropical storms"],
    },
    {"reportDate": "2015-09-16",
     "impactSubtypes" : "Residential Buildings",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactAnnotation" : ['An aerial survey was conducted on 30 August 2015 in the northern districts to observe the flooding situation and the potential damage to housing, agriculture, and infrastructure and to map the scale of population displacement.'],
     "country" : "Bangladesh",
     "location" : ['northern districts'],
     "locationAnnotation" : ['An aerial survey was conducted on 30 August 2015 in the northern districts to observe the flooding situation and the potential damage to housing, agriculture, and infrastructure and to map the scale of population displacement.'],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardType" : ["Flood", "Tropical storms"],
    },
    {"reportDate": "2015-09-16",
     "impactSubtypes" : "Displaced People",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactAnnotation" : ['An aerial survey was conducted on 30 August 2015 in the northern districts to observe the flooding situation and the potential damage to housing, agriculture, and infrastructure and to map the scale of population displacement.'],
     "country" : "Bangladesh",
     "location" : ['northern districts'],
     "locationAnnotation" : ['An aerial survey was conducted on 30 August 2015 in the northern districts to observe the flooding situation and the potential damage to housing, agriculture, and infrastructure and to map the scale of population displacement.'],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardType" : ["Flood", "Tropical storms"],
    },
    {"reportDate": "2015-09-16",
     "impactSubtypes" : "Agriculture",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactAnnotation" : ['An aerial survey was conducted on 30 August 2015 in the northern districts to observe the flooding situation and the potential damage to housing, agriculture, and infrastructure and to map the scale of population displacement.'],
     "country" : "Bangladesh",
     "location" : ['northern districts'],
     "locationAnnotation" : ['An aerial survey was conducted on 30 August 2015 in the northern districts to observe the flooding situation and the potential damage to housing, agriculture, and infrastructure and to map the scale of population displacement.'],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardType" : ["Flood", "Tropical storms"],
    },
    {"reportDate": "2015-09-16",
     "impactSubtypes" : "Agriculture",
     "impactValue" : None, 
     "impactUnit" : None, 
     "impactAnnotation" : ['Floods have caused extensive damage to crops in different parts of the country.'],
     "country" : "Bangladesh",
     "location" : None,
     "locationAnnotation" : None,
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardType" : ["Flood"],
    }
]

In [31]:
df_impact_alldf_impact_list = []
for k,v in labelled_impact_reports_dict_new_haz.items():
    df_impact = pd.DataFrame(v)
    df_impact['appealCode'] = k
    df_impact_list.append(df_impact)
df_impact_all = pd.concat(df_impact_list)
df_impact_all.reset_index(inplace=True, drop=True)

#Save impact csv 
fn = "labelled_reports.csv" ##### CHANGE THE NAME OF THE REPORT TO SAVE
df_impact_all.to_csv(DATA_LABELLED+fn, index=False)